# 02 — Feature Engineering Deep-Dive

This notebook walks through every feature construction step, demonstrating **anti-leakage discipline**: every feature uses only data available *before* the prediction point.

**Feature Categories:**
1. Rolling game stats (win%, margin, scoring volatility) — `.shift(1)`
2. Prior-season team quality (SRS, ratings, Four Factors) — season N-1 only
3. Roster quality (BPM, VORP, depth) — aggregated from player data
4. Matchup differentials (home - away for key features)

In [1]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from nba_predict.data.loader import load_schedules, load_teams_advanced
from nba_predict.features.game_features import build_game_features, get_game_feature_columns
from nba_predict.features.team_features import build_prior_season_features
from nba_predict.features.player_features import build_roster_quality
from nba_predict.features.matchup_features import build_matchup_dataset, get_feature_columns

sns.set_theme(style='whitegrid', palette='deep', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 6)
print('Libraries loaded.')

Libraries loaded.


## 1. Rolling Game Features

Built from `schedules.csv`. Every rolling stat uses `.shift(1)` so the current game's outcome never leaks into its own features.

In [2]:
sched = load_schedules()
sched = build_game_features(sched)

game_cols = get_game_feature_columns()
print(f"Rolling game features: {len(game_cols)}")
for col in game_cols:
    print(f"  {col}")

Rolling game features: 49
  win_pct_season
  pts_avg_season
  pts_allowed_avg_season
  margin_avg_season
  prev_streak
  rest_days
  is_b2b
  season_game_pct
  ot_recent
  is_home
  game_num
  win_pct_last5
  pts_avg_last5
  pts_allowed_last5
  margin_avg_last5
  pts_std_last5
  margin_std_last5
  win_pct_last10
  pts_avg_last10
  pts_allowed_last10
  margin_avg_last10
  pts_std_last10
  margin_std_last10
  win_pct_last20
  pts_avg_last20
  pts_allowed_last20
  margin_avg_last20
  pts_std_last20
  margin_std_last20
  sentiment_score_prev
  sentiment_score_last5
  sentiment_score_last10
  sentiment_score_last20
  sentiment_volume_prev
  sentiment_volume_last5
  sentiment_volume_last10
  sentiment_volume_last20
  sentiment_std_prev
  sentiment_std_last5
  sentiment_std_last10
  sentiment_std_last20
  sentiment_engagement_prev
  sentiment_engagement_last5
  sentiment_engagement_last10
  sentiment_engagement_last20
  sentiment_pos_ratio_prev
  sentiment_pos_ratio_last5
  sentiment_pos_rati

In [3]:
# Verify shift(1): first game of each team-season should have NaN features
first_games = sched.groupby(['team', 'season']).first()
nan_pct = first_games[game_cols].isnull().mean()
print("NaN rate for first game of each team-season (should be ~100%):")
print(nan_pct.describe())

NaN rate for first game of each team-season (should be ~100%):
count    49.000000
mean      0.391431
std       0.476235
min       0.000000
25%       0.000000
50%       0.000000
75%       0.959006
max       0.963975
dtype: float64


In [4]:
# Rolling win% for two teams (2025 season)
teams_to_plot = ['BOS', 'WAS']
season_to_plot = 2025

fig, ax = plt.subplots(figsize=(14, 5))
for team in teams_to_plot:
    team_data = sched[(sched['team'] == team) & (sched['season'] == season_to_plot)]
    if len(team_data) > 0:
        ax.plot(team_data['game_num'], team_data['win_pct_season'], label=team, linewidth=2)

ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Game Number')
ax.set_ylabel('Season Win %')
ax.set_title(f'Rolling Season Win % \u2014 {season_to_plot}')
ax.legend()
plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41023/3647930309.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Rest days impact on win rate
rest_win = sched.dropna(subset=['rest_days']).copy()
rest_win['rest_bucket'] = rest_win['rest_days'].clip(upper=5).astype(int)
rest_wr = rest_win.groupby('rest_bucket')['win'].mean()

fig, ax = plt.subplots(figsize=(8, 5))
rest_wr.plot(kind='bar', ax=ax, color='teal', alpha=0.8)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5)
ax.set_title('Win Rate by Rest Days')
ax.set_xlabel('Rest Days (capped at 5)')
ax.set_ylabel('Win Rate')
plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41023/575984354.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Prior-Season Team Features

Stats from season N-1 joined onto games in season N.

In [6]:
prior_stats = build_prior_season_features()
print(f"Prior-season features: {len(prior_stats.columns) - 2} stats")
print(f"Team-seasons: {len(prior_stats)}")
prior_stats.head()

Prior-season features: 25 stats
Team-seasons: 805


,team,season,off_rtg_prev,def_rtg_prev,net_rtg_prev,pace_prev,ts_pct_prev,srs_prev,sos_prev,mov_prev,...,opp_ft_rate_prev,fg_pct_prev,fg3_pct_prev,ft_pct_prev,trb_prev,ast_prev,stl_prev,blk_prev,tov_prev,pts_prev
0,ATL,2001,102.0,107.9,-5.9,91.7,0.503,-5.41,-0.04,-5.38,...,0.196,0.441,0.317,0.743,45.3,18.9,6.1,5.6,15.4,94.3
1,ATL,2002,98.7,104.3,-5.6,91.9,0.500,-5.55,-0.34,-5.21,...,0.249,0.431,0.357,0.759,42.9,19.0,7.7,4.7,16.7,91.0
2,ATL,2003,101.8,106.4,-4.6,91.9,0.517,-4.41,-0.18,-4.23,...,0.218,0.439,0.354,0.765,41.5,20.2,8.1,4.3,15.5,94.0
3,ATL,2004,102.3,106.1,-3.8,91.0,0.527,-3.87,-0.31,-3.56,...,0.221,0.444,0.352,0.793,42.6,20.5,7.5,5.8,16.7,94.1
4,ATL,2005,101.0,106.1,-5.1,90.8,0.514,-5.00,-0.36,-4.65,...,0.234,0.433,0.335,0.776,42.7,20.1,7.6,5.0,16.5,92.8


In [7]:
# SRS predictive power: prior-season SRS vs current-season wins
teams_adv = load_teams_advanced()
teams_adv['wins'] = pd.to_numeric(teams_adv['wins'], errors='coerce')

srs_col = [c for c in prior_stats.columns if 'srs' in c.lower()]
if srs_col:
    srs_col = srs_col[0]
    srs_df = prior_stats[['team', 'season', srs_col]].merge(
        teams_adv[['team', 'season', 'wins']], on=['team', 'season'], how='inner'
    )
    srs_df[srs_col] = pd.to_numeric(srs_df[srs_col], errors='coerce')
    srs_df = srs_df.dropna(subset=[srs_col, 'wins'])

    fig, ax = plt.subplots(figsize=(8, 8))
    ax.scatter(srs_df[srs_col], srs_df['wins'], alpha=0.3, s=15)
    z = np.polyfit(srs_df[srs_col], srs_df['wins'], 1)
    x_line = np.linspace(srs_df[srs_col].min(), srs_df[srs_col].max(), 100)
    ax.plot(x_line, np.polyval(z, x_line), 'r-', linewidth=2)
    r_sq = srs_df[[srs_col, 'wins']].corr().iloc[0, 1] ** 2
    ax.set_title(f'Prior-Season SRS vs Current Wins (R\u00b2 = {r_sq:.3f})')
    ax.set_xlabel('Prior Season SRS')
    ax.set_ylabel('Current Season Wins')
    plt.tight_layout()
    plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41023/633787088.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Roster Quality Features

In [8]:
roster_qual = build_roster_quality()
roster_cols = [c for c in roster_qual.columns if c not in ('team', 'season')]
print(f"Roster quality features: {roster_cols}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'roster_top5_bpm' in roster_qual.columns:
    axes[0].hist(roster_qual['roster_top5_bpm'].dropna(), bins=40, color='steelblue', alpha=0.7)
    axes[0].set_title('Distribution: Top-5 Player BPM (per team)')
    axes[0].set_xlabel('Average BPM of Top 5 Players')

if 'roster_total_vorp' in roster_qual.columns:
    axes[1].hist(roster_qual['roster_total_vorp'].dropna(), bins=40, color='teal', alpha=0.7)
    axes[1].set_title('Distribution: Total Team VORP')
    axes[1].set_xlabel('Sum of Player VORP')

plt.tight_layout()
plt.show()

Roster quality features: ['roster_top5_bpm', 'roster_top5_per', 'roster_total_vorp', 'roster_total_ws', 'roster_best_bpm', 'roster_depth_bpm', 'roster_size', 'roster_continuity', 'roster_avg_exp', 'roster_avg_height']


/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41023/2479754046.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Matchup Construction

The final step: pair home/away games, join all features, compute differentials.

In [9]:
print("Building full matchup dataset (~30s)...")
matchups = build_matchup_dataset()
feature_cols = get_feature_columns(matchups)

print(f"\nTotal matchups: {len(matchups):,}")
print(f"Total features: {len(feature_cols)}")
print(f"\nFeature categories:")
print(f"  home_*: {sum(1 for c in feature_cols if c.startswith('home_'))}")
print(f"  away_*: {sum(1 for c in feature_cols if c.startswith('away_'))}")
print(f"  diff_*: {sum(1 for c in feature_cols if c.startswith('diff_'))}")

Building full matchup dataset (~30s)...



Total matchups: 31,285
Total features: 200

Feature categories:
  home_*: 84
  away_*: 84
  diff_*: 32


In [10]:
# Top 20 features by correlation with home_win
corrs = matchups[feature_cols + ['home_win']].apply(pd.to_numeric, errors='coerce').corr()['home_win'].drop('home_win')
top20 = corrs.abs().nlargest(20)

fig, ax = plt.subplots(figsize=(10, 8))
top20_vals = corrs[top20.index]
colors = ['#2196F3' if v > 0 else '#F44336' for v in top20_vals]
top20_vals.plot(kind='barh', ax=ax, color=colors)
ax.set_title('Top 20 Features by Correlation with Home Win')
ax.set_xlabel('Pearson Correlation')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

/var/folders/k4/_twnc0ms1vs0f5jj5d4lhbjr0000gn/T/ipykernel_41023/2402354967.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Summary

| Category | Count | Source | Anti-Leakage |
|----------|-------|--------|-------------|
| Rolling game stats | ~30 per side | schedules.csv | `.shift(1)` |
| Prior-season team | ~25 per side | teams_advanced/per_game | Season N-1 only |
| Roster quality | ~5 per side | players_advanced | Season N-1 only |
| Differentials | ~24 | Computed | home - away |
| **Total** | **~153** | | |

**Key insight:** Differential features (home - away) are the strongest predictors.